In [1]:
!pip install transformers==4.52.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 89.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 87.4 MB/s eta 0:00:00:00:01
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "mistralai/Mistral-Nemo-Instruct-2407"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00005.safetensors:   0%|          | 0.00/4.87G [00:00<?, ?B/s]

model-00002-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00003-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [3]:
def generate_text(prompt, max_length=100, num_return_sequences=1):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
    )
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

In [4]:
question = "When was Andrew Samer born?"
answer = generate_text(question)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:2479: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


In [5]:
print(answer[0])

When was Andrew Samer born? Andrew Samer was born on 1986-01-22.

## When did Andrew Samer die? You know how Andrew Samer died? According to our data, Andrew Samer died on unknown (as far as we know).

Andrew Samer was 35 years old when died.


In [6]:
!pip install sentence-transformers PyPDF2 faiss-cpu

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 85.4 MB/s eta 0:00:00:00:0100:01


In [7]:
from sentence_transformers import SentenceTransformer
from PyPDF2 import PdfReader
import faiss

In [8]:
def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    full_text = ""
    for page in reader.pages:
        full_text += page.extract_text() + "\n"
    return full_text

In [9]:
def chunk_text(text, chunk_size=500, overlap=50):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

In [10]:
def embed_chunks(chunks, model_name='sentence-transformers/all-MiniLM-L6-v2'):
    model = SentenceTransformer(model_name)
    embeddings = model.encode(chunks, convert_to_numpy = True)
    return model, embeddings

In [11]:
def create_faiss_index(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)
    return index

In [20]:
def search_index(query, model, index, chunks, k=5):
    query_embedding = model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, k)
    return [chunks[i] for i in indices[0]]

In [13]:
pdf_path = "/kaggle/input/datasets/void6639/doc-paper/Design_and_Implementation_of_PID_Controller_in_Pro.pdf"

text = extract_text_from_pdf(pdf_path)
chunks = chunk_text(text, chunk_size=100, overlap=10)

model_embeddings, embeddings = embed_chunks(chunks)

index = create_faiss_index(embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [21]:
question = "where is MUHAMMAD SHARFI NAJIB from?"
top_chunks = search_index(question, model_embeddings, index, chunks, k=3)

for i, chunk in enumerate(top_chunks, 1):
    print(f"\n--- Chunk {i} ---\n{chunk}")


--- Chunk 1 ---
Design and Implementation of PI D Controller in Programmable Logic Controller for DC Motor Position Control of the Conveyor System MUHAMMAD SHARFI NAJIB, MOHD SHAWAL JADI N, RAJA M. TAUFIKA RAJA ISMAIL, MOHD RUSLLIM MOHAMED Faculty of Electrical & Electronics Engineering, Universiti Malaysia Pahang, Karung Berkunci 12, 25000 Kuantan, Pahang, MALAYSIA E-mail: sharfi@ump.edu.my Abstract: - Direct Current (DC) motor position control using Programmable Logic Controller (PLC) is one of the applications which ar e widely used in automation industr ies. The aim of this project is to implement a Proportional Integral Derivative (PID ) controller in a DC mo tor

--- Chunk 2 ---
response of a practical control system usually ex hibits damped oscillation before reaching steady state. As for DC motor, having a high overshoot is an undesired condition since the starting current is very high. The position of single phase DC Motor of a conveyor system has been controlled by using PID

In [22]:
chunk = top_chunks[0]

prompt = f"Answer the next question: {question} by reading the following text:{chunk}"

In [23]:
answer = generate_text(prompt, max_length=700)
print(answer[0])

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:2479: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


Answer the next question: where is MUHAMMAD SHARFI NAJIB from? by reading the following text:Design and Implementation of PI D Controller in Programmable Logic Controller for DC Motor Position Control of the Conveyor System MUHAMMAD SHARFI NAJIB, MOHD SHAWAL JADI N, RAJA M. TAUFIKA RAJA ISMAIL, MOHD RUSLLIM MOHAMED Faculty of Electrical & Electronics Engineering, Universiti Malaysia Pahang, Karung Berkunci 12, 25000 Kuantan, Pahang, MALAYSIA E-mail: sharfi@ump.edu.my Abstract: - Direct Current (DC) motor position control using Programmable Logic Controller (PLC) is one of the applications which ar e widely used in automation industr ies. The aim of this project is to implement a Proportional Integral Derivative (PID ) controller in a DC mo tor to control the position of a conveyor system in a factory. The controller is designed using PLC programming software, and the results are analyzed using MATLAB/Simulink simulation.

Based on the provided text, Muhammad Sharfi Najib is from Univer